# Phase 2 — FinBERT Sentiment Analysis
**Goal:** Run every TSLA headline through FinBERT and produce one daily sentiment score per trading day.

**What is FinBERT?**
FinBERT is a version of BERT (a transformer model by Google) that was fine-tuned specifically on financial text — earnings reports, analyst notes, financial news. It understands phrases like 'beats expectations' or 'margin compression' much better than a general-purpose model.

We use it as a **black box**: give it a headline → it returns three probabilities: positive, negative, neutral.

## Step 1 — Install libraries

In [ ]:
!pip install transformers torch pandas matplotlib --quiet
print('Done!')

## Step 2 — Import libraries

In [ ]:
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from transformers import BertTokenizer, BertForSequenceClassification
from torch.nn.functional import softmax
import warnings, os
warnings.filterwarnings('ignore')

# Check if GPU is available (Colab gives you a free T4 GPU)
# GPU makes FinBERT inference ~10x faster
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print('If this says cpu, go to Runtime → Change runtime type → T4 GPU')

## Step 3 — Load FinBERT from HuggingFace

HuggingFace is a platform that hosts thousands of pre-trained models. We download FinBERT with just two lines.

First time this runs it downloads ~440MB — subsequent runs use the cached version.

In [ ]:
MODEL_NAME = 'ProsusAI/finbert'

print('Loading FinBERT tokenizer...')
# Tokenizer converts text into numbers the model understands
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

print('Loading FinBERT model...')
# The actual neural network
model = BertForSequenceClassification.from_pretrained(MODEL_NAME)
model = model.to(device)   # move to GPU if available
model.eval()               # set to inference mode (disables dropout)

# FinBERT label mapping: 0=positive, 1=negative, 2=neutral
LABELS = {0: 'positive', 1: 'negative', 2: 'neutral'}

print('\nFinBERT loaded successfully!')
print(f'Labels: {LABELS}')

## Step 4 — Test FinBERT on a few example headlines

Before running on all 1165 headlines, let's verify it works and understand the output format.

In [ ]:
def get_sentiment(headline):
    """
    Takes a single headline string.
    Returns dict with positive, negative, neutral scores (they sum to 1.0)
    and the dominant label.
    """
    # Step 1: Tokenize — convert text to token IDs
    # max_length=512 is BERT's limit; truncation=True handles longer texts
    inputs = tokenizer(
        headline,
        return_tensors='pt',     # return PyTorch tensors
        truncation=True,
        max_length=512,
        padding=True
    ).to(device)

    # Step 2: Run through model (no_grad = don't compute gradients, saves memory)
    with torch.no_grad():
        outputs = model(**inputs)

    # Step 3: Convert raw logits to probabilities using softmax
    # logits are raw scores; softmax squashes them to [0,1] summing to 1
    probs = softmax(outputs.logits, dim=1).squeeze().cpu().numpy()

    return {
        'positive': round(float(probs[0]), 4),
        'negative': round(float(probs[1]), 4),
        'neutral':  round(float(probs[2]), 4),
        'label':    LABELS[probs.argmax()],
        'compound': round(float(probs[0] - probs[1]), 4)  # pos - neg
    }


# Test on example headlines
test_headlines = [
    'Tesla beats Q4 earnings expectations with record deliveries',
    'Tesla stock crashes after Elon Musk sells billions in shares',
    'Tesla announces new Gigafactory location in Mexico',
    'Tesla recalls 2 million vehicles over autopilot safety concerns',
    'Tesla shares rise after Morgan Stanley upgrades to overweight',
]

print(f'{"Headline":<55} {"Pos":>6} {"Neg":>6} {"Neu":>6} {"Label"}')
print('-' * 90)
for h in test_headlines:
    s = get_sentiment(h)
    print(f'{h[:54]:<55} {s["positive"]:>6.3f} {s["negative"]:>6.3f} {s["neutral"]:>6.3f} {s["label"]}')

## Step 5 — Run FinBERT on all 1165 headlines

We process headlines in batches of 16 for speed.
On GPU this takes ~2 minutes. On CPU ~15 minutes.

In [ ]:
# Load news data
os.chdir('/content')
news_df = pd.read_csv('tsla_news_raw.csv')
print(f'Loaded {len(news_df)} headlines')
news_df.head(3)

In [ ]:
def get_sentiment_batch(headlines, batch_size=16):
    """
    Process a list of headlines in batches.
    Much faster than processing one at a time.
    """
    all_results = []

    for i in range(0, len(headlines), batch_size):
        batch = headlines[i : i + batch_size]

        # Tokenize the whole batch at once
        inputs = tokenizer(
            batch,
            return_tensors='pt',
            truncation=True,
            max_length=512,
            padding=True       # pad shorter headlines to same length
        ).to(device)

        with torch.no_grad():
            outputs = model(**inputs)

        probs = softmax(outputs.logits, dim=1).cpu().numpy()

        for p in probs:
            all_results.append({
                'positive': round(float(p[0]), 4),
                'negative': round(float(p[1]), 4),
                'neutral':  round(float(p[2]), 4),
                'label':    LABELS[p.argmax()],
                'compound': round(float(p[0] - p[1]), 4)
            })

        # Progress update every 100 headlines
        if (i + batch_size) % 100 == 0 or (i + batch_size) >= len(headlines):
            print(f'  Processed {min(i+batch_size, len(headlines))}/{len(headlines)} headlines...')

    return all_results


print('Running FinBERT on all headlines...')
headlines_list = news_df['headline'].tolist()
results = get_sentiment_batch(headlines_list)

# Add results back to dataframe
sentiment_df = news_df.copy()
sentiment_df['positive'] = [r['positive'] for r in results]
sentiment_df['negative'] = [r['negative'] for r in results]
sentiment_df['neutral']  = [r['neutral']  for r in results]
sentiment_df['label']    = [r['label']    for r in results]
sentiment_df['compound'] = [r['compound'] for r in results]

print(f'\nDone! Shape: {sentiment_df.shape}')
sentiment_df.head(10)

## Step 6 — Check sentiment distribution

In [ ]:
print('=== Sentiment label distribution ===')
counts = sentiment_df['label'].value_counts()
print(counts)
print(f'\nPositive: {counts.get("positive",0)/len(sentiment_df)*100:.1f}%')
print(f'Negative: {counts.get("negative",0)/len(sentiment_df)*100:.1f}%')
print(f'Neutral:  {counts.get("neutral",0)/len(sentiment_df)*100:.1f}%')

# Plot distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

colors = ['#2ecc71', '#e74c3c', '#95a5a6']
counts.plot(kind='bar', ax=axes[0], color=colors, edgecolor='white')
axes[0].set_title('Headline sentiment counts')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=0)

sentiment_df['compound'].hist(bins=40, ax=axes[1], color='steelblue', edgecolor='white')
axes[1].axvline(0, color='red', linestyle='--', alpha=0.7)
axes[1].set_title('Compound score distribution (pos - neg)')
axes[1].set_xlabel('Compound score')

plt.tight_layout()
plt.savefig('sentiment_distribution.png', dpi=150)
plt.show()

## Step 7 — Aggregate to daily sentiment scores

We have multiple headlines per day. We need ONE score per day to align with price data.

Strategy: take the mean of all scores for that day. This is simple and works well.

In [ ]:
daily_sentiment = sentiment_df.groupby('date').agg(
    positive  = ('positive', 'mean'),
    negative  = ('negative', 'mean'),
    neutral   = ('neutral',  'mean'),
    compound  = ('compound', 'mean'),
    n_headlines = ('headline', 'count')   # how many headlines that day
).reset_index()

print(f'Daily sentiment scores: {len(daily_sentiment)} days')
print(f'\nSample:')
daily_sentiment.head(10)

## Step 8 — Merge sentiment with price data

In [ ]:
price_df = pd.read_csv('tsla_price_data.csv')
print(f'Price data shape: {price_df.shape}')

# Merge on date — left join keeps all trading days
combined = price_df.merge(daily_sentiment, left_on='Date', right_on='date', how='left')
combined.drop(columns=['date'], inplace=True)

# Days with no news get forward-filled sentiment
# (assume sentiment carries over from previous day)
sentiment_cols = ['positive', 'negative', 'neutral', 'compound', 'n_headlines']
combined[sentiment_cols] = combined[sentiment_cols].fillna(method='ffill')

# Fill any remaining NaN at the very start with neutral values
combined['positive'].fillna(0.33, inplace=True)
combined['negative'].fillna(0.33, inplace=True)
combined['neutral'].fillna(0.34, inplace=True)
combined['compound'].fillna(0.0,  inplace=True)
combined['n_headlines'].fillna(0,  inplace=True)

print(f'Combined dataset shape: {combined.shape}')
print(f'Missing values: {combined.isnull().sum().sum()}')
combined.tail(5)

## Step 9 — Visualise sentiment vs price

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
fig.suptitle('TSLA Price vs News Sentiment (2020–2024)', fontsize=14)

dates = pd.to_datetime(combined['Date'])

# Plot 1: Close price
axes[0].plot(dates, combined['Close'], color='#1f77b4', linewidth=1.2)
axes[0].set_ylabel('Close Price (USD)')
axes[0].grid(True, alpha=0.3)
axes[0].set_title('TSLA Closing Price')

# Plot 2: Compound sentiment score
colors = ['#2ecc71' if c >= 0 else '#e74c3c' for c in combined['compound']]
axes[1].bar(dates, combined['compound'], color=colors, alpha=0.7, width=1)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_ylabel('Compound Sentiment')
axes[1].set_xlabel('Date')
axes[1].grid(True, alpha=0.3)
axes[1].set_title('Daily News Sentiment (green=positive, red=negative)')
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.xticks(rotation=45)

plt.tight_layout()
plt.savefig('price_vs_sentiment.png', dpi=150)
plt.show()
print('Chart saved!')

## Step 10 — Save final combined dataset

In [ ]:
combined.to_csv('tsla_combined.csv', index=False)

print('Saved: tsla_combined.csv')
print(f'Shape: {combined.shape}')
print(f'\nFinal columns:')
for col in combined.columns:
    print(f'  {col}')

## Summary

You now have `tsla_combined.csv` — a single dataset with:
- All price + technical indicator features from Phase 1a
- Daily FinBERT sentiment scores merged by date

This is the complete feature set your model will train on.

**Next — Phase 3:** Build the LSTM model in PyTorch that takes this data and predicts next-day price direction.